In [1]:
import modal

In [9]:
image = (
    modal.Image.debian_slim(python_version="3.10")
    .apt_install("libgl1", "libglib2.0-0")
    .pip_install_from_requirements('/Users/so/Desktop/trees/requirements.txt')
)

app = modal.App("mortalitree", image=image)

# Persists the dataset across runs so we don't re-download every time
volume = modal.Volume.from_name("mot", create_if_missing=True)
DATA_DIR = "/data"

In [ ]:
@app.function(volumes={DATA_DIR: volume}, timeout=10_000)
def download_treeboxes():
    from milliontrees.datasets.TreeBoxes import TreeBoxesDataset
    ds = TreeBoxesDataset(root_dir=DATA_DIR, download=True, include_unsupervised=True)
    volume.commit()
    return {"n_images": len(ds), "data_dir": DATA_DIR}

In [13]:
with app.run():
    result = download_treeboxes.remote()

result

FunctionTimeoutError: Task's current input in-01KT3AR12C0K589FSWZZ4B6XYF:1780375880782-0 hit its timeout of 3600s

In [ ]:
# Sanity check: list what landed in the volume
@app.function(volumes={DATA_DIR: volume})
def list_data():
    import os
    for root, dirs, files in os.walk(DATA_DIR):
        depth = root.replace(DATA_DIR, "").count(os.sep)
        if depth > 2:
            continue
        print(root, "->", len(files), "files")

with app.run():
    list_data.remote()